import sys
from pathlib import Path
# El intérprete que ejecuta ESTE notebook. Si la ruta no contiene .venv,
# el kernel es el equivocado: Kernel -> Change Kernel.
print("Intérprete:", sys.executable)
# La carpeta desde la que se ejecuta. Las rutas relativas del notebook
# se resuelven desde aquí, no desde donde está el archivo .ipynb.
print("Carpeta de trabajo:", Path.cwd())
# Las librerías del proyecto, con su versión
for lib in ["numpy", "pandas", "matplotlib", "sklearn"]:
    try:
        print(f" {lib:12}", __import__(lib).__version__)
    except ImportError:
        print(f" {lib:12} NO DISPONIBLE en este kernel")

# Problematica
Se trabaja con el conjunto de datos oficial "tb_patentes_aguas_december_2025_data.csv", que consolida las
patentes por no uso de derechos de aprovechamiento de aguas en Chile (46.467 registros, 21 variables; Tesorería
General de la República, 2025). <br>
## Pregunta de análisis: ¿cómo se distribuyen la morosidad y el saldo impago de estas
patentes según Tesorería Regional, tipo de titular (persona natural o jurídica) y año de emisión? 
<br>
## Objetivo metodológico: 
que el análisis sea trazable y reproducible mediante Git/GitHub y Jupyter Notebooks.

## Subtítulo
### Sección
Texto normal con **negrita**, *cursiva* y `código en línea`.
Probando cambios



### Herramientas del ecosistema científico

| Pieza | Por que se usa | 
|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones de las librerías del proyecto. | 
| JupyterLab + `ipykernel` | Ejecutar el cuaderno con el intérprete del proyecto. | 
| `requirements.txt` | Declarar las dependencias exactas. | 
| Git | Registrar la historia del trabajo y su autoría. | 
| GitHub | Colaborar y publicar la evidencia evaluada. | 
| `pathlib` | Rutas relativas independientes del sistema operativo. | 

En esta fase se usan sobre todo módulos de la biblioteca estándar: el cuaderno construye y
verifica el entorno, todavía no analiza datos. `numpy` y `pandas` aparecen únicamente para
comprobar que están disponibles y para escribir las tablas de documentación.

In [ ]:
import sys                        # intérprete en uso: es lo que se verifica más abajo
import json                       # exportación de metadatos legibles
import platform                   # sistema operativo, para dejarlo en la bitácora
import subprocess                 # consultas a Git desde el cuaderno
import shutil                     # localizar ejecutables (git) sin suponer que existen
import importlib                  # importar el módulo que este cuaderno va a escribir
import importlib.util             # comprobar si un paquete está instalado
from pathlib import Path          # manejo de rutas independiente del sistema operativo
from datetime import date

# Verificación del entorno: instala en el intérprete del kernel lo que falte.
print("Intérprete del kernel:", sys.executable)

REQUERIDOS = {"numpy": "numpy", "pandas": "pandas"}   # módulo: paquete pip
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()

import numpy as np                # presente para verificar el entorno del proyecto
import pandas as pd               # tablas de documentación de la fase

print("numpy", np.__version__, "· pandas", pd.__version__)

# Reproducibilidad: la misma semilla que usa el cuaderno de la Fase 2.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 ·", date.today().isoformat())

## 1. Definición del problema

Deja el problema escrito como una **estructura de datos**

In [ ]:
PROYECTO = {
    "titulo": "consolidacion de patentes por no uso de derechos de aprovechamiento de aguas",
    "grupo": "Grupo 9",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Christian Abarca Araya", "Vicente Agosin Orellana", "Patricia Campos Tapia","Daniel Valdivia Tapia"],
    "problematica": (
        

        
        "Análisis de la morosidad en las patentes por no uso de derechos de aprovechamiento de aguas en Chile, a partir del "
        "conjunto de datos oficial publicado en el Portal de Datos Abiertos del Gobierno de Chile. El objetivo es describir cómo "
        "se distribuyen la morosidad y el saldo impago según Tesorería Regional, tipo de titular (persona natural o jurídica) y "
        "año de emisión, con un flujo de trabajo trazable y reproducible. "
    ),
    "objetivo_general": (
        "Construir un flujo reproducible de análisis que, a partir de datos de morosidad en las patentes por no aprovechamiento de agua "
        "en Chile, entender la distribución de morosidad segun region y tipo de titular"
    ),
    "objetivos_especificos": [
        "Definir el problema y establecer el entorno reproducible del proyecto (F1).",
        "Construir el pipeline de obtención, limpieza y transformación de datos (F2).",
        "Implementar el núcleo algorítmico con programación estructurada y POO (F3).",
        "Comunicar los hallazgos mediante visualización y un informe técnico (F4).",
    ],
    "preguntas": [
        "En qué región se concentran las mayores morosidades por no aprovechamiento de aguas",
        "Qué tipo de titular concentra mayor morosidad por no aprovechamiento de aguas",
    ],
    "criterios_exito": [
        "El repositorio se clona y el cuaderno corre completo sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los cuatro integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": ["Definición del problema", "Entorno reproducible", "Selección del conjunto"],
        "excluye": ["Limpieza e imputación", "Modelado predictivo", "Despliegue"],
    },
    "limitaciones": [
        "Los datos se asumen como verdaderos: No se comprueba su veracidad",
        "La variable objetivo está desbalanceada, lo que condiciona la evaluación posterior.",
        "El conjunto no incluye variables temporales, por lo que no hay análisis de evolución.",
    ],
}

In [ ]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y con un mensaje que dice QUE falta, no solo que algo falló.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

In [ ]:
print("Alcance de la Fase 1")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nLimitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba, con evidencia y no por confianza, que el cuaderno se
está ejecutando con el intérprete del proyecto y que las librerías declaradas están
disponibles en él.

In [ ]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "sklearn"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# se instala scikit-learn, pero se importa sklearn.
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. Intérprete que ejecuta ESTE cuaderno. Si la ruta no contiene .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde aquí,
    #    no desde donde está guardado el archivo .ipynb.
    reporte["carpeta_trabajo"] = str(Path.cwd())
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Librerías del proyecto, con su versión.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se interrumpe el cuaderno: se informa qué instalar y con qué nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO = verificar_entorno(DEPENDENCIAS)